In [1]:
#Cell 1:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils import load_config, set_seed
from src.data.loader import load_source_dataset
from src.preprocessing.cleaning import (
    clean_dataset, normalize_labels, encode_labels, report_missing_and_infinite
)

cfg = load_config()
set_seed(cfg["project"]["random_state"])

df_raw = load_source_dataset(cfg, verbose=True)

  Loaded Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
    shape = 225,745 rows x 80 cols | memory = 149.2 MB

  COMBINED: 225,745 rows x 80 cols | 149.2 MB


In [2]:
#Cell 2 — diagnostic (before cleaning):
missing_inf = report_missing_and_infinite(df_raw, label_col=cfg["dataset"]["label_column"])
print(missing_inf)

                n_missing  n_infinite
Flow Packets/s          0          34
Flow Bytes/s            4          30


In [3]:
#Cell 3 — run cleaning:
df_clean, clean_report = clean_dataset(df_raw, label_col=cfg["dataset"]["label_column"], verbose=True)

DATA CLEANING REPORT
Rows at start:                     225,745
Columns at start:                  80
----------------------------------------------------------------
Columns dropped upfront:           ['source_file', 'Fwd Header Length.1']
Duplicate rows removed:            2,633
Infinite values found:             56
Rows removed (NaN/inf):            30
Rows removed (Flow Duration < 0):  2
----------------------------------------------------------------
Zero-variance columns detected (10):
    - Bwd Avg Bulk Rate
    - Bwd Avg Bytes/Bulk
    - Bwd Avg Packets/Bulk
    - Bwd PSH Flags
    - Bwd URG Flags
    - CWE Flag Count
    - Fwd Avg Bulk Rate
    - Fwd Avg Bytes/Bulk
    - Fwd Avg Packets/Bulk
    - Fwd URG Flags
Matches EDA expectation:           True
----------------------------------------------------------------
Final rows:                        223,080
Final columns (incl. label):       68
Data retained:                     98.819%


In [4]:
#Cell 4 — normalize + encode labels:
df_clean = normalize_labels(df_clean, label_col=cfg["dataset"]["label_column"])
df_clean = encode_labels(df_clean, cfg, label_col=cfg["dataset"]["label_column"])

print("\nBinary label check:")
print(df_clean["label_binary"].value_counts())

Label distribution after cleaning:
    DDOS            128,014   (57.38%)
    BENIGN           95,066   (42.62%)

Binary label check:
label_binary
1    128014
0     95066
Name: count, dtype: int64


In [5]:
#Cell 5 — save:
import json

parquet_path = os.path.join(cfg["paths"]["processed_dir"], "cicids2017_ddos_cleaned.parquet")
df_clean.to_parquet(parquet_path, index=False)

report_path = os.path.join(cfg["paths"]["reports_dir"], "cleaning_report.json")
with open(report_path, "w") as f:
    json.dump(clean_report, f, indent=2)

print("Saved cleaned data to:", parquet_path)
print("Saved cleaning report to:", report_path)
print("Final shape:", df_clean.shape)

Saved cleaned data to: d:\MSc_Project\Network_IDS_Framework\results/processed\cicids2017_ddos_cleaned.parquet
Saved cleaning report to: d:\MSc_Project\Network_IDS_Framework\results/reports\cleaning_report.json
Final shape: (223080, 69)
